In [1]:
import pandas as pd
import networkx as nx
from community.community_louvain import best_partition, modularity
import numpy as np
from collections import defaultdict
import warnings
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')

def load_data(train_file, test_file):
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    return train_df, test_df

def create_graph(train_df):
    G = nx.Graph()
    edges = list(zip(train_df['Node1'], train_df['Node2']))
    G.add_edges_from(edges)
    return G

def get_best_partition(graph, resolution=1.0, n_runs=20):
    def run_partition(_):
        partition = best_partition(graph, resolution=resolution, random_state=None)
        return partition, modularity(partition, graph)
    
    with ThreadPoolExecutor() as executor:
        results = list(executor.map(run_partition, range(n_runs)))
    
    best_part, best_mod = max(results, key=lambda x: x[1])
    return best_part, best_mod

def merge_small_communities(partition, graph, min_size=5):
    community_sizes = defaultdict(int)
    for node, comm_id in partition.items():
        community_sizes[comm_id] += 1
    
    small_comms = {comm_id for comm_id, size in community_sizes.items() if size < min_size}
    if not small_comms:
        return partition
    
    new_partition = partition.copy()
    for node in graph.nodes():
        if partition[node] in small_comms:
            neighbor_comms = defaultdict(int)
            for neighbor in graph.neighbors(node):
                if partition[neighbor] not in small_comms:
                    neighbor_comms[partition[neighbor]] += 1
            if neighbor_comms:
                new_comm = max(neighbor_comms.items(), key=lambda x: x[1])[0]
                new_partition[node] = new_comm
    
    return new_partition

def calculate_node_similarity(G, node_pairs):
    # Pre-compute neighbor sets and node degrees
    all_nodes = set()
    for node1, node2 in node_pairs:
        all_nodes.add(node1)
        all_nodes.add(node2)
    
    neighbor_sets = {node: set(G.neighbors(node)) for node in all_nodes}
    node_degrees = {node: len(neighbors) for node, neighbors in neighbor_sets.items()}
    
    # Pre-compute clustering coefficients
    clustering_coeffs = nx.clustering(G, nodes=all_nodes)
    
    # Pre-compute betweenness centrality for important nodes
    important_nodes = {node for node in all_nodes if node_degrees[node] > np.mean(list(node_degrees.values()))}
    betweenness = nx.betweenness_centrality(G, k=min(1000, len(G.nodes())))
    
    similarities = np.zeros(len(node_pairs))
    
    for i, (node1, node2) in enumerate(node_pairs):
        try:
            # Get neighbor sets
            neighbors1 = neighbor_sets[node1]
            neighbors2 = neighbor_sets[node2]
            
            if not neighbors1 or not neighbors2:
                continue
            
            # Calculate Jaccard similarity
            intersection = len(neighbors1 & neighbors2)
            union = len(neighbors1 | neighbors2)
            jaccard = intersection / union if union > 0 else 0
            
            # Calculate common neighbors ratio
            common_neighbors = intersection
            max_neighbors = max(len(neighbors1), len(neighbors2))
            common_ratio = common_neighbors / max_neighbors if max_neighbors > 0 else 0
            
            # Calculate resource allocation index
            common_neighbors_set = neighbors1 & neighbors2
            resource_alloc = sum(1 / (node_degrees[n] + 1) for n in common_neighbors_set)
            
            # Calculate preferential attachment score
            pref_attach = (node_degrees[node1] * node_degrees[node2]) / (len(G.nodes()) ** 2)
            
            # Calculate Adamic-Adar index
            adamic_adar = sum(1 / np.log(node_degrees[n] + 1) for n in common_neighbors_set)
            
            # Calculate clustering coefficient similarity
            clust_sim = 1 - abs(clustering_coeffs.get(node1, 0) - clustering_coeffs.get(node2, 0))
            
            # Calculate betweenness centrality similarity
            between_sim = 1 - abs(betweenness.get(node1, 0) - betweenness.get(node2, 0))
            
            # Calculate local clustering coefficient
            local_clust1 = nx.clustering(G, nodes=neighbors1)
            local_clust2 = nx.clustering(G, nodes=neighbors2)
            local_clust_sim = 1 - abs(np.mean(list(local_clust1.values())) - np.mean(list(local_clust2.values())))
            
            # Combine metrics with optimized weights
            similarities[i] = (0.42 * jaccard + 
                   0.18 * common_ratio + 
                   0.22 * resource_alloc + 
                   0.03 * pref_attach +   # Reduced from 0.1
                   0.28 * adamic_adar +   # Increased from 0.05
                   0.04 * clust_sim +
                   0.01 * between_sim +   # Reduced from 0.03
                   0.02 * local_clust_sim)
            
        except:
            continue
    
    return similarities

def predict_community_membership(G, node_pairs, best_partition, similarities):
    predictions = np.zeros(len(node_pairs), dtype=np.int32)
    
    # Pre-compute node importance scores
    node_importance = nx.pagerank(G)
    
    for i, (node1, node2) in enumerate(node_pairs):
        # Check if both nodes exist in the partition
        if node1 in best_partition and node2 in best_partition:
            # Base prediction
            predictions[i] = 1 if best_partition[node1] == best_partition[node2] else 0
            
            # If not in same community, check additional features
            if predictions[i] == 0:
                try:
                    # Check path length with importance weighting
                    path_length = nx.shortest_path_length(G, node1, node2)
                    importance_score = (node_importance.get(node1, 0) + node_importance.get(node2, 0)) / 2
                    
                    if path_length <= 2 and similarities[i] > (0.25 - 0.05 * importance_score):
                        predictions[i] = 1
                except nx.NetworkXNoPath:
                    pass
                
                # Check local structure
                try:
                    neighbors1 = set(G.neighbors(node1))
                    neighbors2 = set(G.neighbors(node2))
                    common_neighbors = neighbors1 & neighbors2
                    
                    if len(common_neighbors) >= 2:
                        # Check if common neighbors are in the same community
                        common_community = all(best_partition[n] == best_partition[list(common_neighbors)[0]] 
                                            for n in common_neighbors)
                        if common_community:
                            predictions[i] = 1
                        
                        # Check if nodes have high similarity with importance weighting
                        similarity_threshold = 0.35 - 0.05 * importance_score
                        if similarities[i] > similarity_threshold:
                            predictions[i] = 1
                            
                        # Check if nodes have strong local connections
                        if len(common_neighbors) >= 3 and similarities[i] > 0.3:
                            predictions[i] = 1
                except:
                    pass
        else:
            # Fallback for nodes not in partition with importance weighting
            importance_score = (node_importance.get(node1, 0) + node_importance.get(node2, 0)) / 2
            if similarities[i] > (0.35 - 0.05 * importance_score):
                predictions[i] = 1
    
    return predictions

def main():
    # File paths
    train_file = r"C:\Users\Dspike\Documents\NTUST\SNet\train.csv"
    test_file = r"C:\Users\Dspike\Documents\NTUST\SNet\test.csv"
    output_file = r"C:\Users\Dspike\Documents\NTUST\SNet\sample_submission.csv"
    
    print("Loading data...")
    train_df, test_df = load_data(train_file, test_file)
    
    print("Creating graph...")
    G = create_graph(train_df)
    
    print("Finding best community partition...")
    resolutions = [ 0.96, 1.0]
    best_overall_partition = None
    best_overall_modularity = -float('inf')
    
    for res in resolutions:
        partition, modularity_score = get_best_partition(G, resolution=res, n_runs=24)
        partition = merge_small_communities(partition, G, min_size=6)
        modularity_score = modularity(partition, G)
        print(f"Resolution {res}: Modularity = {modularity_score}")
        if modularity_score > best_overall_modularity:
            best_overall_modularity = modularity_score
            best_overall_partition = partition
    
    print("Calculating node similarities...")
    node_pairs = list(zip(test_df['Node1'], test_df['Node2']))
    similarities = calculate_node_similarity(G, node_pairs)
    
    print("Making predictions...")
    predictions = predict_community_membership(G, node_pairs, best_overall_partition, similarities)
    
    print("Creating submission file...")
    submission_df = pd.DataFrame({
        'Id': range(len(predictions)),
        'Category': predictions
    })
    submission_df = submission_df[['Id', 'Category']]
    submission_df.to_csv(output_file, index=False)
    
    print(f"Submission file saved to {output_file}")
    print(f"Best modularity achieved: {best_overall_modularity}")

if __name__ == "__main__":
    main()
# THIS CODE RESULTED A KAGGLE SCORE OF 0.73166 

Loading data...
Creating graph...
Finding best community partition...
Resolution 0.96: Modularity = 0.8645775515399376
Resolution 1.0: Modularity = 0.8650277461378215
Calculating node similarities...
Making predictions...
Creating submission file...
Submission file saved to C:\Users\Dspike\Documents\NTUST\SNet\sample_submission.csv
Best modularity achieved: 0.8650277461378215


In [ ]:
import pandas as pd
import networkx as nx
from community.community_louvain import best_partition, modularity
import numpy as np
from collections import defaultdict
import warnings
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')

def load_data(train_file, test_file):
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    return train_df, test_df

def create_graph(train_df):
    G = nx.Graph()
    edges = list(zip(train_df['Node1'], train_df['Node2']))
    G.add_edges_from(edges)
    return G

def get_best_partition(graph, resolution=1.0, n_runs=30):
    best_part = None
    best_mod = -float('inf')
    
    for _ in range(n_runs):
        partition = best_partition(graph, resolution=resolution, random_state=None)
        mod = modularity(partition, graph)
        if mod > best_mod:
            best_mod = mod
            best_part = partition
    return best_part, best_mod

def merge_small_communities(partition, graph, min_size=5):
    community_sizes = defaultdict(int)
    for node, comm_id in partition.items():
        community_sizes[comm_id] += 1
    
    small_comms = {comm_id for comm_id, size in community_sizes.items() if size < min_size}
    if not small_comms:
        return partition
    
    new_partition = partition.copy()
    for node in graph.nodes():
        if partition[node] in small_comms:
            neighbor_comms = defaultdict(int)
            for neighbor in graph.neighbors(node):
                if partition[neighbor] not in small_comms:
                    neighbor_comms[partition[neighbor]] += 1
            if neighbor_comms:
                new_comm = max(neighbor_comms.items(), key=lambda x: x[1])[0]
                new_partition[node] = new_comm
    
    return new_partition

def calculate_node_importance(G):
    try:
        pagerank = nx.pagerank(G)
        degree_cent = nx.degree_centrality(G)
        betweenness = nx.betweenness_centrality(G, k=min(1000, len(G.nodes())))
        
        # Combine different centrality measures
        importance = {}
        for node in G.nodes():
            importance[node] = (0.4 * pagerank.get(node, 0) + 
                              0.4 * degree_cent.get(node, 0) + 
                              0.2 * betweenness.get(node, 0))
        return importance
    except:
        return nx.degree_centrality(G)

def calculate_node_similarity(G, node_pairs):
    # Pre-compute neighbor sets and node degrees
    all_nodes = set()
    for node1, node2 in node_pairs:
        all_nodes.add(node1)
        all_nodes.add(node2)
    
    neighbor_sets = {node: set(G.neighbors(node)) for node in all_nodes}
    node_degrees = {node: len(neighbors) for node, neighbors in neighbor_sets.items()}
    
    # Pre-compute clustering coefficients
    try:
        clustering_coeffs = nx.clustering(G, nodes=all_nodes)
    except:
        clustering_coeffs = {node: 0 for node in all_nodes}
    
    # Pre-compute betweenness centrality for important nodes
    try:
        important_nodes = {node for node in all_nodes if node_degrees[node] > np.mean(list(node_degrees.values()))}
        betweenness = nx.betweenness_centrality(G, k=min(1000, len(G.nodes())))
    except:
        betweenness = {node: 0 for node in all_nodes}
    
    similarities = np.zeros(len(node_pairs))
    
    for i, (node1, node2) in enumerate(node_pairs):
        try:
            # Get neighbor sets
            neighbors1 = neighbor_sets[node1]
            neighbors2 = neighbor_sets[node2]
            
            if not neighbors1 or not neighbors2:
                continue
            
            # Calculate Jaccard similarity
            intersection = len(neighbors1 & neighbors2)
            union = len(neighbors1 | neighbors2)
            jaccard = intersection / union if union > 0 else 0
            
            # Calculate common neighbors ratio
            common_neighbors = intersection
            max_neighbors = max(len(neighbors1), len(neighbors2))
            common_ratio = common_neighbors / max_neighbors if max_neighbors > 0 else 0
            
            # Calculate resource allocation index
            common_neighbors_set = neighbors1 & neighbors2
            resource_alloc = sum(1 / (node_degrees[n] + 1) for n in common_neighbors_set)
            
            # Calculate preferential attachment score
            pref_attach = (node_degrees[node1] * node_degrees[node2]) / (len(G.nodes()) ** 2)
            
            # Calculate Adamic-Adar index
            adamic_adar = sum(1 / np.log(node_degrees[n] + 1) for n in common_neighbors_set)
            
            # Calculate clustering coefficient similarity
            clust_sim = 1 - abs(clustering_coeffs.get(node1, 0) - clustering_coeffs.get(node2, 0))
            
            # Calculate betweenness centrality similarity
            between_sim = 1 - abs(betweenness.get(node1, 0) - betweenness.get(node2, 0))
            
            # Calculate local clustering coefficient
            try:
                local_clust1 = nx.clustering(G, nodes=neighbors1)
                local_clust2 = nx.clustering(G, nodes=neighbors2)
                local_clust_sim = 1 - abs(np.mean(list(local_clust1.values())) - np.mean(list(local_clust2.values())))
            except:
                local_clust_sim = 0
            
            # Calculate Katz similarity
            try:
                katz = nx.katz_centrality_numpy(G)
                katz_sim = 1 - abs(katz.get(node1, 0) - katz.get(node2, 0))
            except:
                katz_sim = 0
            
            # Combine metrics with optimized weights
            similarities[i] = (0.3 * jaccard + 
                             0.2 * common_ratio + 
                             0.15 * resource_alloc + 
                             0.1 * pref_attach + 
                             0.1 * adamic_adar +
                             0.05 * clust_sim +
                             0.05 * between_sim +
                             0.03 * local_clust_sim +
                             0.02 * katz_sim)
            
        except:
            continue
    
    return similarities

def predict_community_membership(G, node_pairs, best_partition, similarities):
    predictions = np.zeros(len(node_pairs), dtype=np.int32)
    
    # Pre-compute node importance scores
    node_importance = calculate_node_importance(G)
    
    # Pre-compute community sizes
    community_sizes = defaultdict(int)
    for node, comm_id in best_partition.items():
        community_sizes[comm_id] += 1
    
    for i, (node1, node2) in enumerate(node_pairs):
        # Check if both nodes exist in the partition
        if node1 in best_partition and node2 in best_partition:
            # Base prediction
            predictions[i] = 1 if best_partition[node1] == best_partition[node2] else 0
            
            # If not in same community, check additional features
            if predictions[i] == 0:
                try:
                    # Check path length with importance weighting
                    path_length = nx.shortest_path_length(G, node1, node2)
                    importance_score = (node_importance.get(node1, 0) + node_importance.get(node2, 0)) / 2
                    
                    # Adjust threshold based on community sizes
                    comm1_size = community_sizes[best_partition[node1]]
                    comm2_size = community_sizes[best_partition[node2]]
                    size_factor = min(comm1_size, comm2_size) / max(comm1_size, comm2_size)
                    
                    if path_length <= 2 and similarities[i] > (0.22 - 0.05 * importance_score + 0.03 * size_factor):
                        predictions[i] = 1
                except nx.NetworkXNoPath:
                    pass
                
                # Check local structure
                try:
                    neighbors1 = set(G.neighbors(node1))
                    neighbors2 = set(G.neighbors(node2))
                    common_neighbors = neighbors1 & neighbors2
                    
                    if len(common_neighbors) >= 2:
                        # Check if common neighbors are in the same community
                        common_community = all(best_partition[n] == best_partition[list(common_neighbors)[0]] 
                                            for n in common_neighbors)
                        if common_community:
                            predictions[i] = 1
                        
                        # Check if nodes have high similarity with importance weighting
                        similarity_threshold = 0.32 - 0.05 * importance_score + 0.03 * size_factor
                        if similarities[i] > similarity_threshold:
                            predictions[i] = 1
                            
                        # Check if nodes have strong local connections
                        if len(common_neighbors) >= 3 and similarities[i] > 0.28:
                            predictions[i] = 1
                            
                        # Check if nodes have high degree similarity
                        degree_sim = 1 - abs(node_degrees[node1] - node_degrees[node2]) / max(node_degrees[node1], node_degrees[node2])
                        if degree_sim > 0.8 and similarities[i] > 0.25:
                            predictions[i] = 1
                except:
                    pass
        else:
            # Fallback for nodes not in partition with importance weighting
            importance_score = (node_importance.get(node1, 0) + node_importance.get(node2, 0)) / 2
            if similarities[i] > (0.32 - 0.05 * importance_score):
                predictions[i] = 1
    
    return predictions

def main():
    # File paths
    train_file = r"C:\Users\Dspike\Documents\NTUST\SNet\train.csv"
    test_file = r"C:\Users\Dspike\Documents\NTUST\SNet\test.csv"
    output_file = r"C:\Users\Dspike\Documents\NTUST\SNet\sample_submission.csv"
    
    print("Loading data...")
    train_df, test_df = load_data(train_file, test_file)
    
    print("Creating graph...")
    G = create_graph(train_df)
    
    print("Finding best community partition...")
    resolutions = [0.85, 0.9, 0.95, 0.98, 1.0, 1.02, 1.05, 1.1]
    best_overall_partition = None
    best_overall_modularity = -float('inf')
    
    for res in resolutions:
        partition, modularity_score = get_best_partition(G, resolution=res, n_runs=30)
        partition = merge_small_communities(partition, G, min_size=5)
        modularity_score = modularity(partition, G)
        print(f"Resolution {res}: Modularity = {modularity_score}")
        if modularity_score > best_overall_modularity:
            best_overall_modularity = modularity_score
            best_overall_partition = partition
    
    print("Calculating node similarities...")
    node_pairs = list(zip(test_df['Node1'], test_df['Node2']))
    similarities = calculate_node_similarity(G, node_pairs)
    
    print("Making predictions...")
    predictions = predict_community_membership(G, node_pairs, best_overall_partition, similarities)
    
    print("Creating submission file...")
    submission_df = pd.DataFrame({
        'Id': range(len(predictions)),
        'Category': predictions
    })
    submission_df = submission_df[['Id', 'Category']]
    submission_df.to_csv(output_file, index=False)
    
    print(f"Submission file saved to {output_file}")
    print(f"Best modularity achieved: {best_overall_modularity}")

if __name__ == "__main__":
    main()

Loading data...
Creating graph...
Finding best community partition...
Resolution 0.85: Modularity = 0.8645050201564984
Resolution 0.9: Modularity = 0.8647386204313212
Resolution 0.95: Modularity = 0.8650698718317147
Resolution 0.98: Modularity = 0.864998756176694
Resolution 1.0: Modularity = 0.8652044963763961
Resolution 1.02: Modularity = 0.8648714113743556
Resolution 1.05: Modularity = 0.8651257305391435
Resolution 1.1: Modularity = 0.8653019594072744
Calculating node similarities...
Making predictions...
Creating submission file...
Submission file saved to C:\Users\Dspike\Documents\NTUST\SNet\sample_submission.csv
Best modularity achieved: 0.8653019594072744


In [ ]:
import pandas as pd
import igraph
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from collections import defaultdict
import warnings
import time
warnings.filterwarnings('ignore')

def load_data(train_file, test_file):
    return pd.read_csv(train_file, usecols=['Node1', 'Node2'], engine='pyarrow'), \
           pd.read_csv(test_file, usecols=['Node1', 'Node2'], engine='pyarrow')

def create_graph(train_df):
    edges = list(zip(train_df['Node1'], train_df['Node2']))  # Fixed tuple indexing
    return igraph.Graph.TupleList(edges, directed=False)

def get_best_partition(graph, resolution=1.0, n_runs=3):
    def run_partition(_):
        partition = graph.community_leiden(objective_function='modularity', resolution=resolution, n_iterations=10)
        partition_dict = {}
        for c, vertices in enumerate(partition):
            for v in vertices:
                partition_dict[v] = c
        mod_score = graph.modularity(partition, weights=None)
        return partition_dict, mod_score
    
    with ThreadPoolExecutor() as executor:
        results = list(executor.map(run_partition, range(n_runs)))
    
    best_part, best_mod = max(results, key=lambda x: x[1])
    return best_part, best_mod

def calculate_node_similarity(G, node_pairs):
    all_nodes = set(np.concatenate([node_pairs[:, 0], node_pairs[:, 1]]))
    neighbor_sets = {v.index: set(G.neighbors(v)) for v in G.vs if v.index in all_nodes}
    node_degrees = {v.index: v.degree() for v in G.vs}
    pagerank = G.pagerank(directed=False, damping=0.85)
    
    features = np.zeros((len(node_pairs), 4))
    for i, (node1, node2) in enumerate(node_pairs):
        if node1 not in neighbor_sets or node2 not in neighbor_sets:
            continue
        neighbors1 = neighbor_sets[node1]
        neighbors2 = neighbor_sets[node2]
        
        intersection = len(neighbors1 & neighbors2)
        union = len(neighbors1 | neighbors2)
        features[i, 0] = intersection / union if union > 0 else 0
        features[i, 1] = sum(1 / (node_degrees.get(n, 1) + 1) for n in neighbors1 & neighbors2)
        features[i, 2] = 1 - abs(pagerank[node1] - pagerank[node2]) if node1 < len(pagerank) and node2 < len(pagerank) else 0
        features[i, 3] = abs(node_degrees.get(node1, 0) - node_degrees.get(node2, 0)) / max(node_degrees.get(node1, 1), node_degrees.get(node2, 1))
    
    return features

def generate_training_data(G, partition):
    community_nodes = defaultdict(list)
    for node, comm_id in partition.items():
        community_nodes[comm_id].append(node)
    
    max_positive_pairs = 10000
    positive_pairs = []
    for comm_id, nodes in community_nodes.items():
        if len(nodes) > 1:
            num_pairs = min(50, len(nodes))
            indices = np.random.choice(len(nodes), size=(num_pairs, 2), replace=True)
            pairs = [(nodes[i], nodes[j]) for i, j in indices if i != j]
            positive_pairs.extend(pairs[:max_positive_pairs - len(positive_pairs)])
        if len(positive_pairs) >= max_positive_pairs:
            break
    
    negative_pairs = []
    communities = list(community_nodes.keys())
    if len(communities) > 1:
        num_negative_pairs = min(len(positive_pairs), max_positive_pairs)
        indices = np.random.choice(len(communities), size=(num_negative_pairs, 2), replace=True)
        negative_pairs = [(np.random.choice(community_nodes[communities[i]]), 
                          np.random.choice(community_nodes[communities[j]])) 
                         for i, j in indices if i != j and communities[i] != communities[j]]
        negative_pairs = negative_pairs[:num_negative_pairs]
    
    all_pairs = positive_pairs + negative_pairs
    labels = [1] * len(positive_pairs) + [0] * len(negative_pairs)
    return np.array(all_pairs), labels

def train_classifier(features, labels):
    X_train, X_val, y_train, y_val = train_test_split(
        features, labels, test_size=0.2, random_state=42, stratify=labels
    )
    
    clf = RandomForestClassifier(
        n_estimators=150, max_depth=12, min_samples_split=10, min_samples_leaf=5,
        random_state=42, n_jobs=-1, class_weight='balanced'
    )
    clf.fit(X_train, y_train)
    
    val_preds = clf.predict(X_val)
    val_probs = clf.predict_proba(X_val)[:, 1]
    print(f"Validation accuracy: {accuracy_score(y_val, val_preds):.4f}")
    print(f"Validation AUC: {roc_auc_score(y_val, val_probs):.4f}")
    print(f"Validation log loss: {log_loss(y_val, val_probs):.4f}")
    print("Feature importances:", clf.feature_importances_)
    return clf

def predict_community_membership(G, node_pairs, partitions, features, classifier):
    community_votes = np.zeros(len(node_pairs))
    community_weights = []
    
    for partition, mod_score in partitions:
        part_predictions = np.array([1 if node1 in partition and node2 in partition and partition[node1] == partition[node2] else 0
                                   for node1, node2 in node_pairs])
        weight = (mod_score + 1) / 2
        community_votes += part_predictions * weight
        community_weights.append(weight)
    
    community_votes /= sum(community_weights)
    ml_predictions = classifier.predict_proba(features)[:, 1]
    
    ensemble_weights = [0.6, 0.4]
    predictions = (ensemble_weights[0] * community_votes + ensemble_weights[1] * ml_predictions) > 0.5
    return predictions.astype(int), ml_predictions

def main():
    train_file = r"C:\Users\Dspike\Documents\NTUST\SNet\train.csv"
    test_file = r"C:\Users\Dspike\Documents\NTUST\SNet\test.csv"
    output_file = r"C:\Users\Dspike\Documents\NTUST\SNet\sample_submission.csv"
    
    start_time = time.time()
    print("Loading data...")
    train_df, test_df = load_data(train_file, test_file)
    
    print("Creating graph...")
    G = create_graph(train_df)
    print(f"Graph created with {G.vcount()} nodes and {G.ecount()} edges")
    
    print("Finding best community partitions...")
    resolutions = [0.96, 1.0]
    partitions = []
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(get_best_partition, G, res, n_runs=3) for res in resolutions]
        partitions.extend(future.result() for future in futures)
    
    for i, (partition, mod_score) in enumerate(partitions):
        print(f"Resolution {resolutions[i]}: Modularity = {mod_score}")
    print(f"Number of communities (best partition): {len(set(partitions[0][0].values()))}")
    
    node_pairs = np.array(list(zip(test_df['Node1'], test_df['Node2'])))
    print("Calculating node features...")
    features = calculate_node_similarity(G, node_pairs)
    
    print("Generating training data...")
    training_pairs, training_labels = generate_training_data(G, partitions[0][0])
    
    print("Calculating training features...")
    training_features = calculate_node_similarity(G, training_pairs)
    
    print("Training classifier...")
    classifier = train_classifier(training_features, training_labels)
    
    print("Making predictions...")
    predictions, probabilities = predict_community_membership(G, node_pairs, partitions, features, classifier)
    
    print("Creating submission file...")
    submission_df = pd.DaQtaFrame({Q
        'Id': range(len(predictions)),
        'Category': predictions  # Use probabilities if competition requires probabilities
    })
    submission_df.to_csv(output_file, index=False)
    
    print(f"Submission file saved to {output_file}")
    print(f"Processing completed in {time.time() - start_time:.2f} seconds")

if __name__ == "__main__":
    main()

Loading data...
Creating graph...
Graph created with 163771 nodes and 398711 edges
Finding best community partitions...
Resolution 0.96: Modularity = 0.8755839317143376
Resolution 1.0: Modularity = 0.8753123927373038
Number of communities (best partition): 234
Calculating node features...
Generating training data...
Calculating training features...
Training classifier...
Validation accuracy: 0.6456
Validation AUC: 0.7049
Validation log loss: 0.5956
Feature importances: [0.33419119 0.19006491 0.26588351 0.20986038]
Making predictions...
Creating submission file...
Submission file saved to C:\Users\Dspike\Documents\NTUST\SNet\sample_submission.csv
Processing completed in 10.92 seconds


In [4]:
import pandas as pd
import networkx as nx
import numpy as np
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import warnings
import time
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

# Configuration
CONFIG = {
    "louvain_resolutions": [0.9, 0.95, 1.0, 1.05, 1.1],
    "louvain_n_runs": 10,
    "community_merge_min_size": 5,
    "similarity_weights": {
        "jaccard": 0.35,
        "common_ratio": 0.25,
        "resource_alloc": 0.15,
        "pref_attach": 0.10,
        "adamic_adar": 0.05,
        "clust_sim": 0.05
    },
    "prediction_thresholds": {
        "path_length_max": 2,
        "base_similarity_A": 0.25,
        "importance_factor_A": 0.05,
        "common_neighbors_min_B": 2,
        "similarity_threshold_B": 0.35,
        "importance_factor_B": 0.05,
        "common_neighbors_min_C": 3,
        "similarity_threshold_C": 0.30,
        "fallback_similarity_D": 0.35,
        "importance_factor_D": 0.05
    }
}

def load_data(train_file, test_file):
    print("Loading data...")
    start_time = time.time()
    train_df = pd.read_csv(train_file, usecols=['Node1', 'Node2'], engine='pyarrow')
    test_df = pd.read_csv(test_file, usecols=['Id', 'Node1', 'Node2'], engine='pyarrow')
    print(f"Data loaded in {time.time() - start_time:.2f}s")
    return train_df, test_df

def create_graph_from_df(df):
    print("Creating graph...")
    start_time = time.time()
    G = nx.Graph()
    G.add_edges_from(zip(df['Node1'], df['Node2']))
    print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges in {time.time() - start_time:.2f}s")
    return G

def get_best_partition(graph, resolution=1.0, n_runs=10, seed=7):
    def run_partition(_):
        communities = nx.algorithms.community.louvain_communities(
            graph, resolution=resolution, threshold=0.0001, seed=seed
        )
        partition = {}
        for comm_id, nodes in enumerate(communities):
            for node in nodes:
                partition[node] = comm_id
        mod_score = nx.algorithms.community.modularity(graph, communities, resolution=resolution)
        return partition, mod_score
    
    with ThreadPoolExecutor() as executor:
        results = list(executor.map(run_partition, range(n_runs)))
    
    best_part, best_mod = max(results, key=lambda x: x[1])
    return best_part, best_mod

def merge_small_communities(partition, graph, min_size=5):
    print(f"Merging communities smaller than {min_size}...")
    start_time = time.time()
    community_nodes = defaultdict(list)
    for node, comm_id in partition.items():
        community_nodes[comm_id].append(node)
    
    small_comms = {comm_id for comm_id, nodes in community_nodes.items() if len(nodes) < min_size}
    if not small_comms:
        print(f"No small communities to merge. Took {time.time() - start_time:.2f}s")
        return partition
    
    new_partition = partition.copy()
    for node in [node for comm_id in small_comms for node in community_nodes[comm_id]]:
        neighbor_comms = defaultdict(int)
        for neighbor in graph.neighbors(node):
            if partition.get(neighbor) not in small_comms:
                neighbor_comms[partition[neighbor]] += 1
        if neighbor_comms:
            new_partition[node] = max(neighbor_comms, key=neighbor_comms.get)
    
    print(f"Merging took {time.time() - start_time:.2f}s")
    return new_partition

def calculate_node_similarity_features(G, node_pairs, config):
    print("Calculating similarity features...")
    start_time = time.time()
    all_nodes = set(np.concatenate([node_pairs[:, 0], node_pairs[:, 1]]))
    nodes_in_G = [node for node in all_nodes if G.has_node(node)]
    
    neighbor_sets = {node: set(G.neighbors(node)) for node in nodes_in_G}
    node_degrees = {node: len(neighbors) for node, neighbors in neighbor_sets.items()}
    clustering_coeffs = nx.clustering(G, nodes=nodes_in_G)
    
    features_list = []
    weights = config["similarity_weights"]
    
    for node1, node2 in node_pairs:
        pair_features = {}
        if not (G.has_node(node1) and G.has_node(node2)):
            pair_features = {key: 0.0 for key in weights}
            features_list.append(pair_features)
            continue
        
        neighbors1 = neighbor_sets.get(node1, set())
        neighbors2 = neighbor_sets.get(node2, set())
        deg1 = node_degrees.get(node1, 0)
        deg2 = node_degrees.get(node2, 0)
        
        common_neighbors = neighbors1 & neighbors2
        intersection_len = len(common_neighbors)
        union_len = len(neighbors1 | neighbors2)
        
        pair_features["jaccard"] = intersection_len / union_len if union_len > 0 else 0
        pair_features["common_ratio"] = intersection_len / max(deg1, deg2) if max(deg1, deg2) > 0 else 0
        pair_features["resource_alloc"] = sum(1 / (node_degrees.get(n, 1) + 1e-6) for n in common_neighbors)
        pair_features["pref_attach"] = (deg1 * deg2) / (G.number_of_edges() + 1e-6) if G.number_of_edges() > 0 else 0
        pair_features["adamic_adar"] = sum(1 / (np.log(node_degrees.get(n, 1) + 1) + 1e-6) for n in common_neighbors)
        pair_features["clust_sim"] = 1 - abs(clustering_coeffs.get(node1, 0) - clustering_coeffs.get(node2, 0))
        
        features_list.append(pair_features)
    
    df_features = pd.DataFrame(features_list)
    weighted_similarities = sum(df_features.get(feature, 0) * weight for feature, weight in weights.items())
    print(f"Similarity features calculated in {time.time() - start_time:.2f}s")
    return weighted_similarities, df_features

def predict_links_with_community_rules(G, node_pairs, partition, similarities, config):
    print("Making predictions...")
    start_time = time.time()
    predictions = np.zeros(len(node_pairs), dtype=np.int32)
    thresholds = config["prediction_thresholds"]
    
    node_importance = nx.pagerank(G, alpha=0.85)
    
    for i, (node1, node2) in enumerate(node_pairs):
        sim_score = similarities[i]
        imp1 = node_importance.get(node1, 0)
        imp2 = node_importance.get(node2, 0)
        avg_importance = (imp1 + imp2) / 2
        
        if node1 in partition and node2 in partition:
            if partition[node1] == partition[node2]:
                predictions[i] = 1
            else:
                try:
                    path_length = nx.shortest_path_length(G, node1, node2)
                    if path_length <= thresholds["path_length_max"] and \
                       sim_score > (thresholds["base_similarity_A"] - thresholds["importance_factor_A"] * avg_importance):
                        predictions[i] = 1
                except (nx.NetworkXNoPath, nx.NodeNotFound):
                    pass
                
                if predictions[i] == 0:
                    neighbors1 = set(G.neighbors(node1)) if G.has_node(node1) else set()
                    neighbors2 = set(G.neighbors(node2)) if G.has_node(node2) else set()
                    common_neighbors = neighbors1 & neighbors2
                    if len(common_neighbors) >= thresholds["common_neighbors_min_B"] and \
                       sim_score > (thresholds["similarity_threshold_B"] - thresholds["importance_factor_B"] * avg_importance):
                        predictions[i] = 1
                    elif len(common_neighbors) >= thresholds["common_neighbors_min_C"] and \
                         sim_score > thresholds["similarity_threshold_C"]:
                        predictions[i] = 1
        else:
            if sim_score > (thresholds["fallback_similarity_D"] - thresholds["importance_factor_D"] * avg_importance):
                predictions[i] = 1
    
    print(f"Predictions made in {time.time() - start_time:.2f}s")
    return predictions

def validate_locally(train_df, partition, config):
    print("Performing local validation...")
    start_time = time.time()
    train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=42)
    G_val = create_graph_from_df(train_split)
    
    val_pairs = np.array(list(zip(val_split['Node1'], val_split['Node2'])))
    true_labels = np.ones(len(val_pairs), dtype=np.int32)  # Edges in train.csv are in same community
    
    similarities, _ = calculate_node_similarity_features(G_val, val_pairs, config)
    predictions = predict_links_with_community_rules(G_val, val_pairs, partition, similarities, config)
    
    accuracy = np.mean(predictions == true_labels)
    print(f"Local validation accuracy: {accuracy:.4f} in {time.time() - start_time:.2f}s")
    return accuracy

def main():
    train_file = r"C:\Users\Dspike\Documents\NTUST\SNet\train.csv"
    test_file = r"C:\Users\Dspike\Documents\NTUST\SNet\test.csv"
    output_file = r"C:\Users\Dspike\Documents\NTUST\SNet\sample_submission.csv"
    
    start_time = time.time()
    train_df, test_df = load_data(train_file, test_file)
    print(f"Train set size: {len(train_df)}, Test set size: {len(test_df)}")
    
    G = create_graph_from_df(train_df)
    
    print("Finding best community partition...")
    best_partition = None
    best_modularity = -float('inf')
    best_config = None
    
    for res in CONFIG["louvain_resolutions"]:
        print(f"Testing resolution: {res}")
        partition, mod_score = get_best_partition(G, resolution=res, n_runs=CONFIG["louvain_n_runs"])
        if not partition:
            print(f"Warning: Empty partition for resolution {res}")
            continue
        partition = merge_small_communities(partition, G, min_size=CONFIG["community_merge_min_size"])
        mod_score = nx.algorithms.community.modularity(G, 
            [{node for node, comm_id in partition.items() if comm_id == i} for i in set(partition.values())])
        print(f"Resolution {res}: Modularity = {mod_score:.4f}")
        
        # Local validation
        accuracy = validate_locally(train_df, partition, CONFIG)
        if mod_score > best_modularity and accuracy > 0.7:  # Threshold to ensure reasonable accuracy
            best_modularity = mod_score
            best_partition = partition
            best_config = {"resolution": res, "min_size": CONFIG["community_merge_min_size"]}
    
    if best_partition is None:
        print("Error: No valid partition found")
        return
    
    print(f"Best partition: Modularity = {best_modularity:.4f}, Config = {best_config}")
    print(f"Number of communities: {len(set(best_partition.values()))}")
    
    test_pairs = np.array(list(zip(test_df['Node1'], test_df['Node2'])))
    similarities, _ = calculate_node_similarity_features(G, test_pairs, CONFIG)
    predictions = predict_links_with_community_rules(G, test_pairs, best_partition, similarities, CONFIG)
    
    print("Creating submission file...")
    submission_df = pd.DataFrame({
        'Id': test_df['Id'],
        'Category': predictions
    })
    submission_df.to_csv(output_file, index=False)
    
    print(f"Submission file saved to {output_file}")
    print(f"Predicted value counts:\n{submission_df['Category'].value_counts()}")
    print(f"Processing completed in {time.time() - start_time:.2f}s")

if __name__ == "__main__":
    main()

Loading data...
Data loaded in 0.01s
Train set size: 398711, Test set size: 1000
Creating graph...
Graph created with 163771 nodes and 398711 edges in 0.97s
Finding best community partition...
Testing resolution: 0.9
Merging communities smaller than 5...
No small communities to merge. Took 0.01s
Resolution 0.9: Modularity = 0.8635
Performing local validation...
Creating graph...
Graph created with 148933 nodes and 318968 edges in 1.53s
Calculating similarity features...
Similarity features calculated in 4281.66s
Making predictions...
Predictions made in 2.45s
Local validation accuracy: 0.9061 in 4285.70s
Testing resolution: 0.95
Merging communities smaller than 5...
No small communities to merge. Took 0.01s
Resolution 0.95: Modularity = 0.8635
Performing local validation...
Creating graph...
Graph created with 148933 nodes and 318968 edges in 0.89s
Calculating similarity features...
Similarity features calculated in 4687.32s
Making predictions...
Predictions made in 2.12s
Local validat

In [7]:
import pandas as pd
import networkx as nx
import numpy as np
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import warnings
import time
from sklearn.model_selection import train_test_split
import random
import matplotlib.pyplot as plt  # For visualization
warnings.filterwarnings('ignore')

# Configuration
CONFIG = {
    "louvain_resolutions": [0.5, 0.75, 1.0, 1.25, 1.5],  # Broader range
    "louvain_n_runs": 5,  # Reduced for speed; adjust as needed
    "community_merge_min_size": 5,
    "similarity_weights": {
        "jaccard": 0.35,
        "common_ratio": 0.25,
        "resource_alloc": 0.15,
        "pref_attach": 0.10,
        "adamic_adar": 0.05,
        "clust_sim": 0.05
    },
    "prediction_thresholds": {
        "path_length_max": 2,
        "base_similarity_A": None,  # Set dynamically
        "importance_factor_A": 0.05,
        "common_neighbors_min_B": 2,
        "similarity_threshold_B": 0.35,
        "importance_factor_B": 0.05,
        "common_neighbors_min_C": 3,
        "similarity_threshold_C": 0.30,
        "fallback_similarity_D": 0.35,
        "importance_factor_D": 0.05
    }
}

def load_data(train_file, test_file):
    print("Loading data...")
    start_time = time.time()
    train_df = pd.read_csv(train_file, usecols=['Node1', 'Node2', 'weight'] if 'weight' in pd.read_csv(train_file, nrows=1).columns else ['Node1', 'Node2'], engine='pyarrow')
    test_df = pd.read_csv(test_file, usecols=['Id', 'Node1', 'Node2'], engine='pyarrow')
    print("Data loaded in {:.2f}s".format(time.time() - start_time))
    return train_df, test_df

def create_graph_from_df(df):
    print("Creating graph...")
    start_time = time.time()
    G = nx.Graph()
    if 'weight' in df.columns:
        G.add_weighted_edges_from(zip(df['Node1'], df['Node2'], df['weight']))
    else:
        G.add_edges_from(zip(df['Node1'], df['Node2']))
    print("Graph created with {} nodes and {} edges in {:.2f}s".format(G.number_of_nodes(), G.number_of_edges(), time.time() - start_time))
    return G

def get_best_partition(graph, resolution=1.0, n_runs=5):
    def run_partition(_):
        communities = nx.algorithms.community.louvain_communities(
            graph, resolution=resolution, threshold=0.0001, seed=random.randint(0, 1000)
        )
        partition = {node: comm_id for comm_id, nodes in enumerate(communities) for node in nodes}
        mod_score = nx.algorithms.community.modularity(graph, communities, resolution=resolution, weight='weight' if graph.edges(data='weight') else None)
        return partition, mod_score
    
    with ThreadPoolExecutor() as executor:
        results = list(executor.map(run_partition, range(n_runs)))
    
    best_part, best_mod = max(results, key=lambda x: x[1])
    return best_part, best_mod

def merge_small_communities(partition, graph, min_size=5):
    print("Merging communities smaller than {}...".format(min_size))
    start_time = time.time()
    community_nodes = defaultdict(list)
    for node, comm_id in partition.items():
        community_nodes[comm_id].append(node)
    
    small_comms = {comm_id for comm_id, nodes in community_nodes.items() if len(nodes) < min_size}
    if not small_comms:
        print("No small communities to merge. Took {:.2f}s".format(time.time() - start_time))
        return partition
    
    new_partition = partition.copy()
    m = sum(weight for _, _, weight in graph.edges(data='weight', default=1)) / 2
    for node in [node for comm_id in small_comms for node in community_nodes[comm_id]]:
        best_comm, best_delta_q = None, -float('inf')
        k_i = sum(weight for _, _, weight in graph.edges(node, data='weight', default=1))
        for neighbor in graph.neighbors(node):
            comm = partition.get(neighbor)
            if comm not in small_comms:
                comm_nodes = community_nodes[comm]
                sum_in = sum(weight for u, v, weight in graph.edges(comm_nodes, data='weight', default=1) if u in comm_nodes and v in comm_nodes)
                sum_tot = sum(weight for _, _, weight in graph.edges(comm_nodes, data='weight', default=1))
                k_i_in = sum(weight for _, v, weight in graph.edges(node, data='weight', default=1) if v in comm_nodes)
                delta_q = ((sum_in + k_i_in) / (2 * m) - ((sum_tot + k_i) / (2 * m))**2) - (sum_in / (2 * m) - (sum_tot / (2 * m))**2 - (k_i / (2 * m))**2)
                if delta_q > best_delta_q:
                    best_delta_q, best_comm = delta_q, comm
        if best_comm is not None:
            new_partition[node] = best_comm
    
    print("Merging took {:.2f}s".format(time.time() - start_time))
    return new_partition

def calculate_node_similarity_features(G, node_pairs, config):
    print("Calculating similarity features...")
    start_time = time.time()
    all_nodes = set(np.concatenate([node_pairs[:, 0], node_pairs[:, 1]]))
    nodes_in_G = [node for node in all_nodes if G.has_node(node)]
    
    neighbor_sets = {node: set(G.neighbors(node)) for node in nodes_in_G}
    node_degrees = {node: len(neighbors) for node, neighbors in neighbor_sets.items()}
    clustering_coeffs = nx.clustering(G, nodes=nodes_in_G)
    
    features_list = []
    weights = config["similarity_weights"]
    
    for node1, node2 in node_pairs:
        pair_features = {}
        if not (G.has_node(node1) and G.has_node(node2)):
            pair_features = {key: 0.0 for key in weights}
            features_list.append(pair_features)
            continue
        
        neighbors1 = neighbor_sets.get(node1, set())
        neighbors2 = neighbor_sets.get(node2, set())
        deg1 = node_degrees.get(node1, 0)
        deg2 = node_degrees.get(node2, 0)
        
        common_neighbors = neighbors1 & neighbors2
        intersection_len = len(common_neighbors)
        union_len = len(neighbors1 | neighbors2)
        
        pair_features["jaccard"] = intersection_len / union_len if union_len > 0 else 0
        pair_features["common_ratio"] = intersection_len / max(deg1, deg2) if max(deg1, deg2) > 0 else 0
        pair_features["resource_alloc"] = sum(1 / (node_degrees.get(n, 1) + 1e-6) for n in common_neighbors)
        pair_features["pref_attach"] = (deg1 * deg2) / (G.number_of_edges() + 1e-6) if G.number_of_edges() > 0 else 0
        pair_features["adamic_adar"] = sum(1 / (np.log(node_degrees.get(n, 1) + 1) + 1e-6) for n in common_neighbors)
        pair_features["clust_sim"] = 1 - abs(clustering_coeffs.get(node1, 0) - clustering_coeffs.get(node2, 0))
        
        features_list.append(pair_features)
    
    df_features = pd.DataFrame(features_list)
    weighted_similarities = sum(df_features.get(feature, 0) * weight for feature, weight in weights.items())
    print("Similarity features calculated in {:.2f}s".format(time.time() - start_time))
    return weighted_similarities, df_features

def predict_links_with_community_rules(G, node_pairs, partition, similarities, config):
    print("Making predictions...")
    start_time = time.time()
    predictions = np.zeros(len(node_pairs), dtype=np.int32)
    thresholds = config["prediction_thresholds"].copy()
    
    # Dynamic threshold based on network properties
    thresholds["base_similarity_A"] = thresholds.get("base_similarity_A", 0.25 * (nx.average_clustering(G) + 0.5))
    
    # Use degree-based importance instead of PageRank
    node_importance = {node: G.degree(node, weight='weight' if G.edges(data='weight') else None) / (G.number_of_nodes() + 1e-6) for node in G.nodes()}
    
    for i, (node1, node2) in enumerate(node_pairs):
        sim_score = similarities[i]
        imp1 = node_importance.get(node1, 0)
        imp2 = node_importance.get(node2, 0)
        avg_importance = (imp1 + imp2) / 2
        
        if node1 in partition and node2 in partition:
            if partition[node1] == partition[node2]:
                predictions[i] = 1
            else:
                try:
                    path_length = nx.shortest_path_length(G, node1, node2, weight='weight' if G.edges(data='weight') else None)
                    if path_length <= thresholds["path_length_max"] and \
                       sim_score > (thresholds["base_similarity_A"] - thresholds["importance_factor_A"] * avg_importance):
                        predictions[i] = 1
                except (nx.NetworkXNoPath, nx.NodeNotFound):
                    pass
                
                if predictions[i] == 0:
                    neighbors1 = set(G.neighbors(node1)) if G.has_node(node1) else set()
                    neighbors2 = set(G.neighbors(node2)) if G.has_node(node2) else set()
                    common_neighbors = neighbors1 & neighbors2
                    if len(common_neighbors) >= thresholds["common_neighbors_min_B"] and \
                       sim_score > (thresholds["similarity_threshold_B"] - thresholds["importance_factor_B"] * avg_importance):
                        predictions[i] = 1
                    elif len(common_neighbors) >= thresholds["common_neighbors_min_C"] and \
                         sim_score > thresholds["similarity_threshold_C"]:
                        predictions[i] = 1
        else:
            if sim_score > (thresholds["fallback_similarity_D"] - thresholds["importance_factor_D"] * avg_importance):
                predictions[i] = 1
    
    print("Predictions made in {:.2f}s".format(time.time() - start_time))
    return predictions

def validate_locally(train_df, partition, config):
    print("Performing local validation...")
    start_time = time.time()
    train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=42)
    G_val = create_graph_from_df(train_split)
    
    val_pairs = np.array(list(zip(val_split['Node1'], val_split['Node2'])))
    true_labels = np.ones(len(val_pairs), dtype=np.int32)  # Positive examples
    
    # Sample negative examples
    all_nodes = list(G_val.nodes())
    negative_pairs = []
    for _ in range(len(val_pairs)):
        n1, n2 = random.sample(all_nodes, 2)
        while G_val.has_edge(n1, n2):
            n1, n2 = random.sample(all_nodes, 2)
        negative_pairs.append((n1, n2))
    negative_pairs = np.array(negative_pairs)
    true_labels = np.concatenate([true_labels, np.zeros(len(negative_pairs), dtype=np.int32)])
    val_pairs = np.concatenate([val_pairs, negative_pairs])
    
    similarities, _ = calculate_node_similarity_features(G_val, val_pairs, config)
    predictions = predict_links_with_community_rules(G_val, val_pairs, partition, similarities, config)
    
    accuracy = np.mean(predictions == true_labels)
    print("Local validation accuracy: {:.4f} in {:.2f}s".format(accuracy, time.time() - start_time))
    return accuracy

def visualize_communities(partition, resolutions, mod_scores):
    # Community size distribution
    community_nodes = defaultdict(list)
    for node, comm_id in partition.items():
        community_nodes[comm_id].append(node)
    sizes = [len(nodes) for nodes in community_nodes.values()]
    
    plt.figure(figsize=(10, 6))
    plt.hist(sizes, bins=30, color='skyblue', edgecolor='black')
    plt.xlabel('Community Size')
    plt.ylabel('Number of Communities')
    plt.title('Community Size Distribution')
    plt.savefig('community_sizes.png')
    plt.close()
    
    # Modularity vs. Resolution
    plt.figure(figsize=(10, 6))
    plt.plot(resolutions, mod_scores, marker='o', color='teal')
    plt.xlabel('Resolution')
    plt.ylabel('Modularity')
    plt.title('Modularity vs. Resolution')
    plt.savefig('modularity_vs_resolution.png')
    plt.close()

def main():
    train_file = r"C:\Users\Dspike\Documents\NTUST\SNet\train.csv"
    test_file = r"C:\Users\Dspike\Documents\NTUST\SNet\test.csv"
    output_file = r"C:\Users\Dspike\Documents\NTUST\SNet\sample_submission.csv"
    
    start_time = time.time()
    train_df, test_df = load_data(train_file, test_file)
    print("Train set size: {}, Test set size: {}".format(len(train_df), len(test_df)))
    
    G = create_graph_from_df(train_df)
    
    print("Finding best community partition...")
    best_partition = None
    best_modularity = -float('inf')
    best_config = None
    mod_scores = []
    tested_resolutions = []
    
    for res in CONFIG["louvain_resolutions"]:
        print("Testing resolution: {}".format(res))
        partition, mod_score = get_best_partition(G, resolution=res, n_runs=CONFIG["louvain_n_runs"])
        if not partition:
            print("Warning: Empty partition for resolution {}".format(res))
            continue
        partition = merge_small_communities(partition, G, min_size=CONFIG["community_merge_min_size"])
        communities = [{node for node, comm_id in partition.items() if comm_id == i} for i in set(partition.values())]
        mod_score = nx.algorithms.community.modularity(G, communities, weight='weight' if G.edges(data='weight') else None)
        print("Resolution {}: Modularity = {:.4f}".format(res, mod_score))
        
        accuracy = validate_locally(train_df, partition, CONFIG)
        mod_scores.append(mod_score)
        tested_resolutions.append(res)
        
        if mod_score > best_modularity and accuracy > 0.7:
            best_modularity = mod_score
            best_partition = partition
            best_config = {"resolution": res, "min_size": CONFIG["community_merge_min_size"]}
    
    if best_partition is None:
        print("Error: No valid partition found")
        return
    
    print("Best partition: Modularity = {:.4f}, Config = {}".format(best_modularity, best_config))
    print("Number of communities: {}".format(len(set(best_partition.values()))))
    
    # Visualize communities
    visualize_communities(best_partition, tested_resolutions, mod_scores)
    
    # Save community assignments
    pd.DataFrame.from_dict(best_partition, orient='index', columns=['Community']).to_csv('communities.csv')
    
    test_pairs = np.array(list(zip(test_df['Node1'], test_df['Node2'])))
    similarities, _ = calculate_node_similarity_features(G, test_pairs, CONFIG)
    predictions = predict_links_with_community_rules(G, test_pairs, best_partition, similarities, CONFIG)
    
    print("Creating submission file...")
    submission_df = pd.DataFrame({
        'Id': test_df['Id'],
        'Category': predictions
    })
    submission_df.to_csv(output_file, index=False)
    
    print("Submission file saved to {}".format(output_file))
    print("Predicted value counts:\n{}".format(submission_df['Category'].value_counts()))
    print("Processing completed in {:.2f}s".format(time.time() - start_time))

if __name__ == "__main__":
    main()

Loading data...
Data loaded in 0.01s
Train set size: 398711, Test set size: 1000
Creating graph...
Graph created with 163771 nodes and 398711 edges in 1.36s
Finding best community partition...
Testing resolution: 0.5
Merging communities smaller than 5...
No small communities to merge. Took 0.01s
Resolution 0.5: Modularity = 0.8581
Performing local validation...
Creating graph...
Graph created with 148933 nodes and 318968 edges in 0.81s
Calculating similarity features...


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
from community import best_partition, modularity
from node2vec import Node2Vec
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import f1_score, roc_auc_score
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm




# Configuration
#DATA_PATH = "your_dataset_path/"
TRAIN_FILE =  r"C:\Users\Dspike\Documents\NTUST\SNet\train.csv"
TEST_FILE =  r"C:\Users\Dspike\Documents\NTUST\SNet\test.csv"
MODEL_SAVE_PATH = "xgboost_model.json"

def load_data():
    """Load and merge train/test data"""
    train = pd.read_csv(TRAIN_FILE)
    test = pd.read_csv(TEST_FILE)
    return train, test

def create_graph(df):
    """Build undirected graph with edge weights"""
    G = nx.Graph()
    edges = [(row['Node1'], row['Node2'], 1.0) for _, row in df.iterrows()]
    G.add_weighted_edges_from(edges)
    return G

def louvain_communities(G, n_runs=20):
    """Parallel Louvain with resolution tuning"""
    def run_partition(resolution):
        partition = best_partition(G, resolution=resolution, random_state=np.random.randint(1e6))
        return partition, modularity(partition, G)
    
    resolutions = [0.9, 0.95, 1.0, 1.05, 1.1]
    with ThreadPoolExecutor() as executor:
        results = list(tqdm(executor.map(run_partition, resolutions), total=len(resolutions)))
    
    best_part, best_mod = max(results, key=lambda x: x[1])
    print(f"Best modularity: {best_mod:.4f} at resolution {resolutions[results.index((best_part, best_mod))]}")
    return best_part

def generate_node2vec_embeddings(G, dimensions=64):
    """Generate Node2Vec embeddings, handling missing nodes"""
    # Initialize Node2Vec with min_count=0 to include all nodes
    n2v = Node2Vec(G, dimensions=dimensions, walk_length=30, num_walks=100, workers=3)
    model = n2v.fit(window=10, min_count=0)  # Set min_count=0 to include all nodes
    # Create embeddings dictionary, using zero vector for missing nodes
    default_embedding = np.zeros(dimensions)  # Default embedding for nodes not in vocabulary
    embeddings = {}
    for node in G.nodes():
        try:
            embeddings[node] = model.wv[str(node)]  # Ensure node is string, as Node2Vec converts nodes to strings
        except KeyError:
            embeddings[node] = default_embedding  # Assign zero vector for missing nodes
    return embeddings
def extract_features(G, node_pairs, communities, embeddings):
    """Feature engineering pipeline"""
    features = []
    for node1, node2 in tqdm(node_pairs):
        # Community features
        same_comm = int(communities.get(node1, -1) == int(communities.get(node2, -1)))
        comm_size1 = sum(1 for n in communities if communities[n] == communities.get(node1, -1))
        comm_size2 = sum(1 for n in communities if communities[n] == communities.get(node2, -1))
        
        # Structural features
        neighbors1 = set(G.neighbors(node1)) if node1 in G else set()
        neighbors2 = set(G.neighbors(node2)) if node2 in G else set()
        common_neighbors = neighbors1 & neighbors2
        jaccard = len(common_neighbors) / len(neighbors1 | neighbors2) if (neighbors1 or neighbors2) else 0
        adamic_adar = sum(1/np.log(len(set(G.neighbors(n)))) for n in common_neighbors if G.has_node(n))
        
        # Embedding features
        emb_sim = np.dot(embeddings.get(node1, np.zeros(64)), embeddings.get(node2, np.zeros(64))) / (
                   np.linalg.norm(embeddings.get(node1, np.zeros(64))) * np.linalg.norm(embeddings.get(node2, np.zeros(64))) + 1e-9)
        
        features.append([
            same_comm,
            comm_size1,
            comm_size2,
            jaccard,
            adamic_adar,
            len(common_neighbors),
            G.degree(node1) if node1 in G else 0,
            G.degree(node2) if node2 in G else 0,
            emb_sim
        ])
    
    return np.array(features)

def train_xgboost(X_train, y_train,X_val, y_val):
    """Train optimized XGBoost classifier"""
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'max_depth': 6,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'seed': 42,
        'n_jobs': -1
    }
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        evals=[(dtrain, 'train'), (dval, 'eval')],
        early_stopping_rounds=50,
        verbose_eval=20
    )
    model.save_model(MODEL_SAVE_PATH)
    return model

def main():
    # Data pipeline
    print("Loading data...")
    train_df, test_df = load_data()
    
    print("Creating graph...")
    G = create_graph(train_df)
    
    print("Detecting communities...")
    communities = louvain_communities(G)
    
    print("Generating Node2Vec embeddings...")
    embeddings = generate_node2vec_embeddings(G)
    
    # Prepare training data (using train edges as positive samples)
    node_pairs = list(zip(train_df['Node1'], train_df['Node2']))
    y = np.ones(len(node_pairs))  # Positive samples
    
    # Generate negative samples
    all_nodes = list(G.nodes())
    neg_pairs = []
    for _ in range(len(node_pairs)):
        n1, n2 = np.random.choice(all_nodes, 2, replace=False)
        while G.has_edge(n1, n2):
            n1, n2 = np.random.choice(all_nodes, 2, replace=False)
        neg_pairs.append((n1, n2))
    
    node_pairs += neg_pairs
    y = np.concatenate([y, np.zeros(len(neg_pairs))])
    
    print("Extracting features...")
    X = extract_features(G, node_pairs, communities, embeddings)
    
    # Train/test split
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    print("Training XGBoost...")
    model = train_xgboost(X_train, y_train,X_val, y_val)
    
    # Validation
    dval = xgb.DMatrix(X_val)
    val_preds = model.predict(dval)
    print(f"\nValidation AUC: {roc_auc_score(y_val, val_preds):.4f}")
    print(f"Validation F1: {f1_score(y_val, val_preds > 0.5):.4f}")
    
    # Generate test predictions
    print("Predicting test set...")
    test_pairs = list(zip(test_df['Node1'], test_df['Node2']))
    X_test = extract_features(G, test_pairs, communities, embeddings)
    dtest = xgb.DMatrix(X_test)
    test_preds = model.predict(dtest)
    
    # Save submission
    submission = pd.DataFrame({
        'Id': range(len(test_preds)),
        'Category': (test_preds > 0.5).astype(int)
    })
    submission.to_csv("submission.csv", index=False)
    print("Submission saved to submission.csv")

if __name__ == "__main__":
    main()

Loading data...
Creating graph...
Detecting communities...


100%|██████████| 5/5 [02:45<00:00, 33.05s/it] 


Best modularity: 0.8640 at resolution 1.0
Generating Node2Vec embeddings...


Computing transition probabilities: 100%|██████████| 163771/163771 [00:44<00:00, 3647.16it/s] 


In [ ]:
import pandas as pd
import networkx as nx
from community.community_louvain import best_partition, modularity
import numpy as np
from collections import defaultdict
import warnings
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')

def load_data(train_file, test_file):
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    return train_df, test_df

def create_graph(train_df):
    G = nx.Graph()
    edges = list(zip(train_df['Node1'], train_df['Node2']))
    G.add_edges_from(edges)
    return G

def get_best_partition(graph, resolution=1.0, n_runs=30):
    best_part = None
    best_mod = -float('inf')
    
    for _ in range(n_runs):
        partition = best_partition(graph, resolution=resolution, random_state=None)
        mod = modularity(partition, graph)
        if mod > best_mod:
            best_mod = mod
            best_part = partition
    return best_part, best_mod

def merge_small_communities(partition, graph, min_size=5):
    community_sizes = defaultdict(int)
    for node, comm_id in partition.items():
        community_sizes[comm_id] += 1
    
    small_comms = {comm_id for comm_id, size in community_sizes.items() if size < min_size}
    if not small_comms:
        return partition
    
    new_partition = partition.copy()
    for node in graph.nodes():
        if partition[node] in small_comms:
            neighbor_comms = defaultdict(int)
            for neighbor in graph.neighbors(node):
                if partition[neighbor] not in small_comms:
                    neighbor_comms[partition[neighbor]] += 1
            if neighbor_comms:
                new_comm = max(neighbor_comms.items(), key=lambda x: x[1])[0]
                new_partition[node] = new_comm
    
    return new_partition

def calculate_node_importance(G):
    try:
        pagerank = nx.pagerank(G)
        degree_cent = nx.degree_centrality(G)
        betweenness = nx.betweenness_centrality(G, k=min(1000, len(G.nodes())))
        
        # Combine different centrality measures
        importance = {}
        for node in G.nodes():
            importance[node] = (0.4 * pagerank.get(node, 0) + 
                              0.4 * degree_cent.get(node, 0) + 
                              0.2 * betweenness.get(node, 0))
        return importance
    except:
        return nx.degree_centrality(G)

def calculate_node_similarity(G, node_pairs):
    # Pre-compute neighbor sets and node degrees
    all_nodes = set()
    for node1, node2 in node_pairs:
        all_nodes.add(node1)
        all_nodes.add(node2)
    
    neighbor_sets = {node: set(G.neighbors(node)) for node in all_nodes}
    node_degrees = {node: len(neighbors) for node, neighbors in neighbor_sets.items()}
    
    # Pre-compute clustering coefficients
    try:
        clustering_coeffs = nx.clustering(G, nodes=all_nodes)
    except:
        clustering_coeffs = {node: 0 for node in all_nodes}
    
    # Pre-compute betweenness centrality for important nodes
    try:
        important_nodes = {node for node in all_nodes if node_degrees[node] > np.mean(list(node_degrees.values()))}
        betweenness = nx.betweenness_centrality(G, k=min(1000, len(G.nodes())))
    except:
        betweenness = {node: 0 for node in all_nodes}
    
    similarities = np.zeros(len(node_pairs))
    
    for i, (node1, node2) in enumerate(node_pairs):
        try:
            # Get neighbor sets
            neighbors1 = neighbor_sets[node1]
            neighbors2 = neighbor_sets[node2]
            
            if not neighbors1 or not neighbors2:
                continue
            
            # Calculate Jaccard similarity
            intersection = len(neighbors1 & neighbors2)
            union = len(neighbors1 | neighbors2)
            jaccard = intersection / union if union > 0 else 0
            
            # Calculate common neighbors ratio
            common_neighbors = intersection
            max_neighbors = max(len(neighbors1), len(neighbors2))
            common_ratio = common_neighbors / max_neighbors if max_neighbors > 0 else 0
            
            # Calculate resource allocation index
            common_neighbors_set = neighbors1 & neighbors2
            resource_alloc = sum(1 / (node_degrees[n] + 1) for n in common_neighbors_set)
            
            # Calculate preferential attachment score
            pref_attach = (node_degrees[node1] * node_degrees[node2]) / (len(G.nodes()) ** 2)
            
            # Calculate Adamic-Adar index
            adamic_adar = sum(1 / np.log(node_degrees[n] + 1) for n in common_neighbors_set)
            
            # Calculate clustering coefficient similarity
            clust_sim = 1 - abs(clustering_coeffs.get(node1, 0) - clustering_coeffs.get(node2, 0))
            
            # Calculate betweenness centrality similarity
            between_sim = 1 - abs(betweenness.get(node1, 0) - betweenness.get(node2, 0))
            
            # Calculate local clustering coefficient
            try:
                local_clust1 = nx.clustering(G, nodes=neighbors1)
                local_clust2 = nx.clustering(G, nodes=neighbors2)
                local_clust_sim = 1 - abs(np.mean(list(local_clust1.values())) - np.mean(list(local_clust2.values())))
            except:
                local_clust_sim = 0
            
            # Calculate Katz similarity
            try:
                katz = nx.katz_centrality_numpy(G)
                katz_sim = 1 - abs(katz.get(node1, 0) - katz.get(node2, 0))
            except:
                katz_sim = 0
            
            # Combine metrics with optimized weights
            similarities[i] = (0.3 * jaccard + 
                             0.2 * common_ratio + 
                             0.15 * resource_alloc + 
                             0.1 * pref_attach + 
                             0.1 * adamic_adar +
                             0.05 * clust_sim +
                             0.05 * between_sim +
                             0.03 * local_clust_sim +
                             0.02 * katz_sim)
            
        except:
            continue
    
    return similarities

def predict_community_membership(G, node_pairs, best_partition, similarities):
    predictions = np.zeros(len(node_pairs), dtype=np.int32)
    
    # Pre-compute node importance scores
    node_importance = calculate_node_importance(G)
    
    # Pre-compute community sizes
    community_sizes = defaultdict(int)
    for node, comm_id in best_partition.items():
        community_sizes[comm_id] += 1
    
    for i, (node1, node2) in enumerate(node_pairs):
        # Check if both nodes exist in the partition
        if node1 in best_partition and node2 in best_partition:
            # Base prediction
            predictions[i] = 1 if best_partition[node1] == best_partition[node2] else 0
            
            # If not in same community, check additional features
            if predictions[i] == 0:
                try:
                    # Check path length with importance weighting
                    path_length = nx.shortest_path_length(G, node1, node2)
                    importance_score = (node_importance.get(node1, 0) + node_importance.get(node2, 0)) / 2
                    
                    # Adjust threshold based on community sizes
                    comm1_size = community_sizes[best_partition[node1]]
                    comm2_size = community_sizes[best_partition[node2]]
                    size_factor = min(comm1_size, comm2_size) / max(comm1_size, comm2_size)
                    
                    if path_length <= 2 and similarities[i] > (0.22 - 0.05 * importance_score + 0.03 * size_factor):
                        predictions[i] = 1
                except nx.NetworkXNoPath:
                    pass
                
                # Check local structure
                try:
                    neighbors1 = set(G.neighbors(node1))
                    neighbors2 = set(G.neighbors(node2))
                    common_neighbors = neighbors1 & neighbors2
                    
                    if len(common_neighbors) >= 2:
                        # Check if common neighbors are in the same community
                        common_community = all(best_partition[n] == best_partition[list(common_neighbors)[0]] 
                                            for n in common_neighbors)
                        if common_community:
                            predictions[i] = 1
                        
                        # Check if nodes have high similarity with importance weighting
                        similarity_threshold = 0.32 - 0.05 * importance_score + 0.03 * size_factor
                        if similarities[i] > similarity_threshold:
                            predictions[i] = 1
                            
                        # Check if nodes have strong local connections
                        if len(common_neighbors) >= 3 and similarities[i] > 0.28:
                            predictions[i] = 1
                            
                        # Check if nodes have high degree similarity
                        degree_sim = 1 - abs(node_degrees[node1] - node_degrees[node2]) / max(node_degrees[node1], node_degrees[node2])
                        if degree_sim > 0.8 and similarities[i] > 0.25:
                            predictions[i] = 1
                except:
                    pass
        else:
            # Fallback for nodes not in partition with importance weighting
            importance_score = (node_importance.get(node1, 0) + node_importance.get(node2, 0)) / 2
            if similarities[i] > (0.32 - 0.05 * importance_score):
                predictions[i] = 1
    
    return predictions

def main():
    # File paths
    train_file = r"C:\Users\Dspike\Documents\NTUST\SNet\train.csv"
    test_file = r"C:\Users\Dspike\Documents\NTUST\SNet\test.csv"
    output_file = r"C:\Users\Dspike\Documents\NTUST\SNet\sample_submission.csv"
    
    print("Loading data...")
    train_df, test_df = load_data(train_file, test_file)
    
    print("Creating graph...")
    G = create_graph(train_df)
    
    print("Finding best community partition...")
    resolutions = [0.85, 0.9, 0.95, 0.98, 1.0, 1.02, 1.05, 1.1]
    best_overall_partition = None
    best_overall_modularity = -float('inf')
    
    for res in resolutions:
        partition, modularity_score = get_best_partition(G, resolution=res, n_runs=30)
        partition = merge_small_communities(partition, G, min_size=5)
        modularity_score = modularity(partition, G)
        print(f"Resolution {res}: Modularity = {modularity_score}")
        if modularity_score > best_overall_modularity:
            best_overall_modularity = modularity_score
            best_overall_partition = partition
    
    print("Calculating node similarities...")
    node_pairs = list(zip(test_df['Node1'], test_df['Node2']))
    similarities = calculate_node_similarity(G, node_pairs)
    
    print("Making predictions...")
    predictions = predict_community_membership(G, node_pairs, best_overall_partition, similarities)
    
    print("Creating submission file...")
    submission_df = pd.DataFrame({
        'Id': range(len(predictions)),
        'Category': predictions
    })
    submission_df = submission_df[['Id', 'Category']]
    submission_df.to_csv(output_file, index=False)
    
    print(f"Submission file saved to {output_file}")
    print(f"Best modularity achieved: {best_overall_modularity}")

if __name__ == "__main__":
    main()

Loading data...
Creating graph...
Finding best community partition...
Resolution 0.85: Modularity = 0.8645050201564984
Resolution 0.9: Modularity = 0.8647386204313212
Resolution 0.95: Modularity = 0.8650698718317147
Resolution 0.98: Modularity = 0.864998756176694
Resolution 1.0: Modularity = 0.8652044963763961
Resolution 1.02: Modularity = 0.8648714113743556
Resolution 1.05: Modularity = 0.8651257305391435
Resolution 1.1: Modularity = 0.8653019594072744
Calculating node similarities...
Making predictions...
Creating submission file...
Submission file saved to C:\Users\Dspike\Documents\NTUST\SNet\sample_submission.csv
Best modularity achieved: 0.8653019594072744


In [ ]:
import pandas as pd
import networkx as nx
from community.community_louvain import best_partition, modularity
import numpy as np
from collections import defaultdict
import warnings
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')

def load_data(train_file, test_file):
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    return train_df, test_df

def create_graph(train_df):
    G = nx.Graph()
    edges = list(zip(train_df['Node1'], train_df['Node2']))
    G.add_edges_from(edges)
    return G

def get_best_partition(graph, resolution=1.0, n_runs=30):
    best_part = None
    best_mod = -float('inf')
    
    for _ in range(n_runs):
        partition = best_partition(graph, resolution=resolution, random_state=None)
        mod = modularity(partition, graph)
        if mod > best_mod:
            best_mod = mod
            best_part = partition
    return best_part, best_mod

def merge_small_communities(partition, graph, min_size=5):
    community_sizes = defaultdict(int)
    for node, comm_id in partition.items():
        community_sizes[comm_id] += 1
    
    small_comms = {comm_id for comm_id, size in community_sizes.items() if size < min_size}
    if not small_comms:
        return partition
    
    new_partition = partition.copy()
    for node in graph.nodes():
        if partition[node] in small_comms:
            neighbor_comms = defaultdict(int)
            for neighbor in graph.neighbors(node):
                if partition[neighbor] not in small_comms:
                    neighbor_comms[partition[neighbor]] += 1
            if neighbor_comms:
                new_comm = max(neighbor_comms.items(), key=lambda x: x[1])[0]
                new_partition[node] = new_comm
    
    return new_partition

def calculate_node_importance(G):
    try:
        pagerank = nx.pagerank(G)
        degree_cent = nx.degree_centrality(G)
        betweenness = nx.betweenness_centrality(G, k=min(1000, len(G.nodes())))
        
        # Combine different centrality measures
        importance = {}
        for node in G.nodes():
            importance[node] = (0.4 * pagerank.get(node, 0) + 
                              0.4 * degree_cent.get(node, 0) + 
                              0.2 * betweenness.get(node, 0))
        return importance
    except:
        return nx.degree_centrality(G)

def calculate_node_similarity(G, node_pairs):
    # Pre-compute neighbor sets and node degrees
    all_nodes = set()
    for node1, node2 in node_pairs:
        all_nodes.add(node1)
        all_nodes.add(node2)
    
    neighbor_sets = {node: set(G.neighbors(node)) for node in all_nodes}
    node_degrees = {node: len(neighbors) for node, neighbors in neighbor_sets.items()}
    
    # Pre-compute clustering coefficients
    try:
        clustering_coeffs = nx.clustering(G, nodes=all_nodes)
    except:
        clustering_coeffs = {node: 0 for node in all_nodes}
    
    # Pre-compute betweenness centrality for important nodes
    try:
        important_nodes = {node for node in all_nodes if node_degrees[node] > np.mean(list(node_degrees.values()))}
        betweenness = nx.betweenness_centrality(G, k=min(1000, len(G.nodes())))
    except:
        betweenness = {node: 0 for node in all_nodes}
    
    similarities = np.zeros(len(node_pairs))
    
    for i, (node1, node2) in enumerate(node_pairs):
        try:
            # Get neighbor sets
            neighbors1 = neighbor_sets[node1]
            neighbors2 = neighbor_sets[node2]
            
            if not neighbors1 or not neighbors2:
                continue
            
            # Calculate Jaccard similarity
            intersection = len(neighbors1 & neighbors2)
            union = len(neighbors1 | neighbors2)
            jaccard = intersection / union if union > 0 else 0
            
            # Calculate common neighbors ratio
            common_neighbors = intersection
            max_neighbors = max(len(neighbors1), len(neighbors2))
            common_ratio = common_neighbors / max_neighbors if max_neighbors > 0 else 0
            
            # Calculate resource allocation index
            common_neighbors_set = neighbors1 & neighbors2
            resource_alloc = sum(1 / (node_degrees[n] + 1) for n in common_neighbors_set)
            
            # Calculate preferential attachment score
            pref_attach = (node_degrees[node1] * node_degrees[node2]) / (len(G.nodes()) ** 2)
            
            # Calculate Adamic-Adar index
            adamic_adar = sum(1 / np.log(node_degrees[n] + 1) for n in common_neighbors_set)
            
            # Calculate clustering coefficient similarity
            clust_sim = 1 - abs(clustering_coeffs.get(node1, 0) - clustering_coeffs.get(node2, 0))
            
            # Calculate betweenness centrality similarity
            between_sim = 1 - abs(betweenness.get(node1, 0) - betweenness.get(node2, 0))
            
            # Calculate local clustering coefficient
            try:
                local_clust1 = nx.clustering(G, nodes=neighbors1)
                local_clust2 = nx.clustering(G, nodes=neighbors2)
                local_clust_sim = 1 - abs(np.mean(list(local_clust1.values())) - np.mean(list(local_clust2.values())))
            except:
                local_clust_sim = 0
            
            # Calculate Katz similarity
            try:
                katz = nx.katz_centrality_numpy(G)
                katz_sim = 1 - abs(katz.get(node1, 0) - katz.get(node2, 0))
            except:
                katz_sim = 0
            
            # Combine metrics with optimized weights
            similarities[i] = (0.3 * jaccard + 
                             0.2 * common_ratio + 
                             0.15 * resource_alloc + 
                             0.1 * pref_attach + 
                             0.1 * adamic_adar +
                             0.05 * clust_sim +
                             0.05 * between_sim +
                             0.03 * local_clust_sim +
                             0.02 * katz_sim)
            
        except:
            continue
    
    return similarities

def predict_community_membership(G, node_pairs, best_partition, similarities):
    predictions = np.zeros(len(node_pairs), dtype=np.int32)
    
    # Pre-compute node importance scores
    node_importance = calculate_node_importance(G)
    
    # Pre-compute community sizes
    community_sizes = defaultdict(int)
    for node, comm_id in best_partition.items():
        community_sizes[comm_id] += 1
    
    for i, (node1, node2) in enumerate(node_pairs):
        # Check if both nodes exist in the partition
        if node1 in best_partition and node2 in best_partition:
            # Base prediction
            predictions[i] = 1 if best_partition[node1] == best_partition[node2] else 0
            
            # If not in same community, check additional features
            if predictions[i] == 0:
                try:
                    # Check path length with importance weighting
                    path_length = nx.shortest_path_length(G, node1, node2)
                    importance_score = (node_importance.get(node1, 0) + node_importance.get(node2, 0)) / 2
                    
                    # Adjust threshold based on community sizes
                    comm1_size = community_sizes[best_partition[node1]]
                    comm2_size = community_sizes[best_partition[node2]]
                    size_factor = min(comm1_size, comm2_size) / max(comm1_size, comm2_size)
                    
                    if path_length <= 2 and similarities[i] > (0.22 - 0.05 * importance_score + 0.03 * size_factor):
                        predictions[i] = 1
                except nx.NetworkXNoPath:
                    pass
                
                # Check local structure
                try:
                    neighbors1 = set(G.neighbors(node1))
                    neighbors2 = set(G.neighbors(node2))
                    common_neighbors = neighbors1 & neighbors2
                    
                    if len(common_neighbors) >= 2:
                        # Check if common neighbors are in the same community
                        common_community = all(best_partition[n] == best_partition[list(common_neighbors)[0]] 
                                            for n in common_neighbors)
                        if common_community:
                            predictions[i] = 1
                        
                        # Check if nodes have high similarity with importance weighting
                        similarity_threshold = 0.32 - 0.05 * importance_score + 0.03 * size_factor
                        if similarities[i] > similarity_threshold:
                            predictions[i] = 1
                            
                        # Check if nodes have strong local connections
                        if len(common_neighbors) >= 3 and similarities[i] > 0.28:
                            predictions[i] = 1
                            
                        # Check if nodes have high degree similarity
                        degree_sim = 1 - abs(node_degrees[node1] - node_degrees[node2]) / max(node_degrees[node1], node_degrees[node2])
                        if degree_sim > 0.8 and similarities[i] > 0.25:
                            predictions[i] = 1
                except:
                    pass
        else:
            # Fallback for nodes not in partition with importance weighting
            importance_score = (node_importance.get(node1, 0) + node_importance.get(node2, 0)) / 2
            if similarities[i] > (0.32 - 0.05 * importance_score):
                predictions[i] = 1
    
    return predictions

def main():
    # File paths
    train_file = r"C:\Users\Dspike\Documents\NTUST\SNet\train.csv"
    test_file = r"C:\Users\Dspike\Documents\NTUST\SNet\test.csv"
    output_file = r"C:\Users\Dspike\Documents\NTUST\SNet\sample_submission.csv"
    
    print("Loading data...")
    train_df, test_df = load_data(train_file, test_file)
    
    print("Creating graph...")
    G = create_graph(train_df)
    
    print("Finding best community partition...")
    resolutions = [0.85, 0.9, 0.95, 0.98, 1.0, 1.02, 1.05, 1.1]
    best_overall_partition = None
    best_overall_modularity = -float('inf')
    
    for res in resolutions:
        partition, modularity_score = get_best_partition(G, resolution=res, n_runs=30)
        partition = merge_small_communities(partition, G, min_size=5)
        modularity_score = modularity(partition, G)
        print(f"Resolution {res}: Modularity = {modularity_score}")
        if modularity_score > best_overall_modularity:
            best_overall_modularity = modularity_score
            best_overall_partition = partition
    
    print("Calculating node similarities...")
    node_pairs = list(zip(test_df['Node1'], test_df['Node2']))
    similarities = calculate_node_similarity(G, node_pairs)
    
    print("Making predictions...")
    predictions = predict_community_membership(G, node_pairs, best_overall_partition, similarities)
    
    print("Creating submission file...")
    submission_df = pd.DataFrame({
        'Id': range(len(predictions)),
        'Category': predictions
    })
    submission_df = submission_df[['Id', 'Category']]
    submission_df.to_csv(output_file, index=False)
    
    print(f"Submission file saved to {output_file}")
    print(f"Best modularity achieved: {best_overall_modularity}")

if __name__ == "__main__":
    main()

Loading data...
Creating graph...
Finding best community partition...
Resolution 0.85: Modularity = 0.8645050201564984
Resolution 0.9: Modularity = 0.8647386204313212
Resolution 0.95: Modularity = 0.8650698718317147
Resolution 0.98: Modularity = 0.864998756176694
Resolution 1.0: Modularity = 0.8652044963763961
Resolution 1.02: Modularity = 0.8648714113743556
Resolution 1.05: Modularity = 0.8651257305391435
Resolution 1.1: Modularity = 0.8653019594072744
Calculating node similarities...
Making predictions...
Creating submission file...
Submission file saved to C:\Users\Dspike\Documents\NTUST\SNet\sample_submission.csv
Best modularity achieved: 0.8653019594072744


In [ ]:
import pandas as pd
import networkx as nx
from community.community_louvain import best_partition, modularity
import numpy as np
from collections import defaultdict
import warnings
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')

def load_data(train_file, test_file):
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    return train_df, test_df

def create_graph(train_df):
    G = nx.Graph()
    edges = list(zip(train_df['Node1'], train_df['Node2']))
    G.add_edges_from(edges)
    return G

def get_best_partition(graph, resolution=1.0, n_runs=30):
    best_part = None
    best_mod = -float('inf')
    
    for _ in range(n_runs):
        partition = best_partition(graph, resolution=resolution, random_state=None)
        mod = modularity(partition, graph)
        if mod > best_mod:
            best_mod = mod
            best_part = partition
    return best_part, best_mod

def merge_small_communities(partition, graph, min_size=5):
    community_sizes = defaultdict(int)
    for node, comm_id in partition.items():
        community_sizes[comm_id] += 1
    
    small_comms = {comm_id for comm_id, size in community_sizes.items() if size < min_size}
    if not small_comms:
        return partition
    
    new_partition = partition.copy()
    for node in graph.nodes():
        if partition[node] in small_comms:
            neighbor_comms = defaultdict(int)
            for neighbor in graph.neighbors(node):
                if partition[neighbor] not in small_comms:
                    neighbor_comms[partition[neighbor]] += 1
            if neighbor_comms:
                new_comm = max(neighbor_comms.items(), key=lambda x: x[1])[0]
                new_partition[node] = new_comm
    
    return new_partition

def calculate_node_importance(G):
    try:
        pagerank = nx.pagerank(G)
        degree_cent = nx.degree_centrality(G)
        betweenness = nx.betweenness_centrality(G, k=min(1000, len(G.nodes())))
        
        # Combine different centrality measures
        importance = {}
        for node in G.nodes():
            importance[node] = (0.4 * pagerank.get(node, 0) + 
                              0.4 * degree_cent.get(node, 0) + 
                              0.2 * betweenness.get(node, 0))
        return importance
    except:
        return nx.degree_centrality(G)

def calculate_node_similarity(G, node_pairs):
    # Pre-compute neighbor sets and node degrees
    all_nodes = set()
    for node1, node2 in node_pairs:
        all_nodes.add(node1)
        all_nodes.add(node2)
    
    neighbor_sets = {node: set(G.neighbors(node)) for node in all_nodes}
    node_degrees = {node: len(neighbors) for node, neighbors in neighbor_sets.items()}
    
    # Pre-compute clustering coefficients
    try:
        clustering_coeffs = nx.clustering(G, nodes=all_nodes)
    except:
        clustering_coeffs = {node: 0 for node in all_nodes}
    
    # Pre-compute betweenness centrality for important nodes
    try:
        important_nodes = {node for node in all_nodes if node_degrees[node] > np.mean(list(node_degrees.values()))}
        betweenness = nx.betweenness_centrality(G, k=min(1000, len(G.nodes())))
    except:
        betweenness = {node: 0 for node in all_nodes}
    
    similarities = np.zeros(len(node_pairs))
    
    for i, (node1, node2) in enumerate(node_pairs):
        try:
            # Get neighbor sets
            neighbors1 = neighbor_sets[node1]
            neighbors2 = neighbor_sets[node2]
            
            if not neighbors1 or not neighbors2:
                continue
            
            # Calculate Jaccard similarity
            intersection = len(neighbors1 & neighbors2)
            union = len(neighbors1 | neighbors2)
            jaccard = intersection / union if union > 0 else 0
            
            # Calculate common neighbors ratio
            common_neighbors = intersection
            max_neighbors = max(len(neighbors1), len(neighbors2))
            common_ratio = common_neighbors / max_neighbors if max_neighbors > 0 else 0
            
            # Calculate resource allocation index
            common_neighbors_set = neighbors1 & neighbors2
            resource_alloc = sum(1 / (node_degrees[n] + 1) for n in common_neighbors_set)
            
            # Calculate preferential attachment score
            pref_attach = (node_degrees[node1] * node_degrees[node2]) / (len(G.nodes()) ** 2)
            
            # Calculate Adamic-Adar index
            adamic_adar = sum(1 / np.log(node_degrees[n] + 1) for n in common_neighbors_set)
            
            # Calculate clustering coefficient similarity
            clust_sim = 1 - abs(clustering_coeffs.get(node1, 0) - clustering_coeffs.get(node2, 0))
            
            # Calculate betweenness centrality similarity
            between_sim = 1 - abs(betweenness.get(node1, 0) - betweenness.get(node2, 0))
            
            # Calculate local clustering coefficient
            try:
                local_clust1 = nx.clustering(G, nodes=neighbors1)
                local_clust2 = nx.clustering(G, nodes=neighbors2)
                local_clust_sim = 1 - abs(np.mean(list(local_clust1.values())) - np.mean(list(local_clust2.values())))
            except:
                local_clust_sim = 0
            
            # Calculate Katz similarity
            try:
                katz = nx.katz_centrality_numpy(G)
                katz_sim = 1 - abs(katz.get(node1, 0) - katz.get(node2, 0))
            except:
                katz_sim = 0
            
            # Combine metrics with optimized weights
            similarities[i] = (0.3 * jaccard + 
                             0.2 * common_ratio + 
                             0.15 * resource_alloc + 
                             0.1 * pref_attach + 
                             0.1 * adamic_adar +
                             0.05 * clust_sim +
                             0.05 * between_sim +
                             0.03 * local_clust_sim +
                             0.02 * katz_sim)
            
        except:
            continue
    
    return similarities

def predict_community_membership(G, node_pairs, best_partition, similarities):
    predictions = np.zeros(len(node_pairs), dtype=np.int32)
    
    # Pre-compute node importance scores
    node_importance = calculate_node_importance(G)
    
    # Pre-compute community sizes
    community_sizes = defaultdict(int)
    for node, comm_id in best_partition.items():
        community_sizes[comm_id] += 1
    
    for i, (node1, node2) in enumerate(node_pairs):
        # Check if both nodes exist in the partition
        if node1 in best_partition and node2 in best_partition:
            # Base prediction
            predictions[i] = 1 if best_partition[node1] == best_partition[node2] else 0
            
            # If not in same community, check additional features
            if predictions[i] == 0:
                try:
                    # Check path length with importance weighting
                    path_length = nx.shortest_path_length(G, node1, node2)
                    importance_score = (node_importance.get(node1, 0) + node_importance.get(node2, 0)) / 2
                    
                    # Adjust threshold based on community sizes
                    comm1_size = community_sizes[best_partition[node1]]
                    comm2_size = community_sizes[best_partition[node2]]
                    size_factor = min(comm1_size, comm2_size) / max(comm1_size, comm2_size)
                    
                    if path_length <= 2 and similarities[i] > (0.22 - 0.05 * importance_score + 0.03 * size_factor):
                        predictions[i] = 1
                except nx.NetworkXNoPath:
                    pass
                
                # Check local structure
                try:
                    neighbors1 = set(G.neighbors(node1))
                    neighbors2 = set(G.neighbors(node2))
                    common_neighbors = neighbors1 & neighbors2
                    
                    if len(common_neighbors) >= 2:
                        # Check if common neighbors are in the same community
                        common_community = all(best_partition[n] == best_partition[list(common_neighbors)[0]] 
                                            for n in common_neighbors)
                        if common_community:
                            predictions[i] = 1
                        
                        # Check if nodes have high similarity with importance weighting
                        similarity_threshold = 0.32 - 0.05 * importance_score + 0.03 * size_factor
                        if similarities[i] > similarity_threshold:
                            predictions[i] = 1
                            
                        # Check if nodes have strong local connections
                        if len(common_neighbors) >= 3 and similarities[i] > 0.28:
                            predictions[i] = 1
                            
                        # Check if nodes have high degree similarity
                        degree_sim = 1 - abs(node_degrees[node1] - node_degrees[node2]) / max(node_degrees[node1], node_degrees[node2])
                        if degree_sim > 0.8 and similarities[i] > 0.25:
                            predictions[i] = 1
                except:
                    pass
        else:
            # Fallback for nodes not in partition with importance weighting
            importance_score = (node_importance.get(node1, 0) + node_importance.get(node2, 0)) / 2
            if similarities[i] > (0.32 - 0.05 * importance_score):
                predictions[i] = 1
    
    return predictions

def main():
    # File paths
    train_file = r"C:\Users\Dspike\Documents\NTUST\SNet\train.csv"
    test_file = r"C:\Users\Dspike\Documents\NTUST\SNet\test.csv"
    output_file = r"C:\Users\Dspike\Documents\NTUST\SNet\sample_submission.csv"
    
    print("Loading data...")
    train_df, test_df = load_data(train_file, test_file)
    
    print("Creating graph...")
    G = create_graph(train_df)
    
    print("Finding best community partition...")
    resolutions = [0.85, 0.9, 0.95, 0.98, 1.0, 1.02, 1.05, 1.1]
    best_overall_partition = None
    best_overall_modularity = -float('inf')
    
    for res in resolutions:
        partition, modularity_score = get_best_partition(G, resolution=res, n_runs=30)
        partition = merge_small_communities(partition, G, min_size=5)
        modularity_score = modularity(partition, G)
        print(f"Resolution {res}: Modularity = {modularity_score}")
        if modularity_score > best_overall_modularity:
            best_overall_modularity = modularity_score
            best_overall_partition = partition
    
    print("Calculating node similarities...")
    node_pairs = list(zip(test_df['Node1'], test_df['Node2']))
    similarities = calculate_node_similarity(G, node_pairs)
    
    print("Making predictions...")
    predictions = predict_community_membership(G, node_pairs, best_overall_partition, similarities)
    
    print("Creating submission file...")
    submission_df = pd.DataFrame({
        'Id': range(len(predictions)),
        'Category': predictions
    })
    submission_df = submission_df[['Id', 'Category']]
    submission_df.to_csv(output_file, index=False)
    
    print(f"Submission file saved to {output_file}")
    print(f"Best modularity achieved: {best_overall_modularity}")

if __name__ == "__main__":
    main()

Loading data...
Creating graph...
Finding best community partition...
Resolution 0.85: Modularity = 0.8645050201564984
Resolution 0.9: Modularity = 0.8647386204313212
Resolution 0.95: Modularity = 0.8650698718317147
Resolution 0.98: Modularity = 0.864998756176694
Resolution 1.0: Modularity = 0.8652044963763961
Resolution 1.02: Modularity = 0.8648714113743556
Resolution 1.05: Modularity = 0.8651257305391435
Resolution 1.1: Modularity = 0.8653019594072744
Calculating node similarities...
Making predictions...
Creating submission file...
Submission file saved to C:\Users\Dspike\Documents\NTUST\SNet\sample_submission.csv
Best modularity achieved: 0.8653019594072744
